# Training Notebook for event_intelligence/train_direct.py

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import psycopg2
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import json
import joblib
import sys
import traceback

# Connection details
DB_DB = "postgres"
DB_USER = "postgres"
DB_PASS = "Madhavan@2004@"
DB_HOST = "db.lpoaswfbufnosziwhtsq.supabase.co"
DB_PORT = "5432"

def main():
    print("--- STARTING EVENT INTELLIGENCE K-FOLD TRAINING ---")
    
    try:
        # 1. Connect
        conn = psycopg2.connect(host=DB_HOST, database=DB_DB, user=DB_USER, password=DB_PASS, port=DB_PORT)
        
        # 2. Extract Data
        print("Extracting sales transactions in chunks...")
        sales_query = "SELECT transaction_date, store_id, product_id, units_sold, promotion_flag FROM public.sales_transactions"
        sales_chunks = pd.read_sql(sales_query, conn, chunksize=25000)
        sales_df = pd.concat(sales_chunks)

        print("Extracting intelligence signals and metadata...")
        events_df = pd.read_sql("SELECT event_id, city_id, start_date, end_date, event_type, severity_level FROM public.events", conn)
        impact_df = pd.read_sql("SELECT event_id, category_id, impact_weight FROM public.event_category_impact", conn)
        stores_df = pd.read_sql("SELECT store_id, city_id FROM public.stores", conn)
        products_df = pd.read_sql("SELECT product_id, category_id FROM public.products", conn)
        conn.close()
        
        # 3. Feature Engineering
        print("Merging and engineering features...")
        sales_df['transaction_date'] = pd.to_datetime(sales_df['transaction_date'])
        events_df['start_date'] = pd.to_datetime(events_df['start_date'])
        events_df['end_date'] = pd.to_datetime(events_df['end_date'])
        
        events_df['event_duration'] = (events_df['end_date'] - events_df['start_date']).dt.days + 1
        
        sales_df['transaction_date'] = sales_df['transaction_date'].dt.date
        events_df['start_date'] = events_df['start_date'].dt.date
        events_df['end_date'] = events_df['end_date'].dt.date
        
        sales_df = sales_df.merge(stores_df, on='store_id')
        sales_df = sales_df.merge(products_df, on='product_id')
        
        baselines = sales_df[sales_df['promotion_flag'] == False].groupby(['store_id', 'category_id'])['units_sold'].mean().reset_index()
        baselines.rename(columns={'units_sold': 'baseline_units'}, inplace=True)
        
        expanded_events = events_df.merge(impact_df, on='event_id').merge(stores_df, on='city_id')
        
        training_df = expanded_events.merge(
            sales_df[sales_df['promotion_flag'] == True],
            left_on=['store_id', 'category_id', 'start_date'],
            right_on=['store_id', 'category_id', 'transaction_date']
        )
        
        training_df = training_df.merge(baselines, on=['store_id', 'category_id'])
        training_df['actual_uplift'] = (training_df['units_sold'] - training_df['baseline_units']) / training_df['baseline_units']
        
        df = training_df[['event_type', 'severity_level', 'event_duration', 'impact_weight', 'actual_uplift']].copy()
        df.rename(columns={'impact_weight': 'expected_weight'}, inplace=True)
        df.dropna(inplace=True)

        le_type, le_sev = LabelEncoder(), LabelEncoder()
        df['event_type_enc'] = le_type.fit_transform(df['event_type'])
        df['severity_level_enc'] = le_sev.fit_transform(df['severity_level'])
        
        joblib.dump(le_type, 'ml/event_intelligence/label_encoder_type.pkl')
        joblib.dump(le_sev, 'ml/event_intelligence/label_encoder_sev.pkl')
        
        features = ['event_type_enc', 'severity_level_enc', 'event_duration', 'expected_weight']
        X = df[features].values
        y = df['actual_uplift'].values
        
        # 4. Train/Test Split (80/20)
        print(f"Total valid samples: {len(X)}")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
        print(f"Training on {len(X_train)} samples, Testing on {len(X_test)} samples...")
        
        # 5. K-Fold Cross Validation
        print("Performing 5-Fold Cross Validation...")
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_mapes = []
        cv_rmses = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            X_tr, y_tr = X_train[train_idx], y_train[train_idx]
            X_val, y_val = X_train[val_idx], y_train[val_idx]
            
            model_cv = xgb.XGBRegressor(
                n_estimators=300, 
                learning_rate=0.03, 
                max_depth=8, 
                objective='reg:squarederror', 
                random_state=42,
                tree_method='hist'
            )
            model_cv.fit(X_tr, y_tr)
            preds = model_cv.predict(X_val)
            
            cv_mapes.append(mean_absolute_percentage_error(y_val, preds))
            cv_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
            
        print(f"CV MAPE: {np.mean(cv_mapes):.4f} (+/- {np.std(cv_mapes):.4f})")
        print(f"CV RMSE: {np.mean(cv_rmses):.4f} (+/- {np.std(cv_rmses):.4f})")

        # 6. Final Model Training
        print("Training final model on full train set...")
        final_model = xgb.XGBRegressor(
            n_estimators=500, 
            learning_rate=0.03, 
            max_depth=8, 
            objective='reg:squarederror', 
            random_state=42,
            tree_method='hist'
        )
        final_model.fit(X_train, y_train)
        
        # 7. Evaluation
        y_pred = final_model.predict(X_test)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        directional_acc = np.mean((y_test > 0) == (y_pred > 0))
        
        report = {
            "model_name": "Event Intelligence K-Fold XGBoost",
            "algorithm": "XGBoost Regressor",
            "training_samples": len(X_train),
            "test_samples": len(X_test),
            "cross_validation_5fold": {
                "mean_MAPE": f"{np.mean(cv_mapes):.4f}",
                "mean_RMSE": f"{np.mean(cv_rmses):.4f}"
            },
            "holdout_test_metrics": {
                "MAPE": f"{mape:.4f}", 
                "RMSE": f"{rmse:.4f}", 
                "Directional_Accuracy": f"{directional_acc:.2%}"
            },
            "feature_importances": dict(zip(features, [float(i) for i in final_model.feature_importances_]))
        }
        
        print("\n--- FINAL REPORT ---")
        print(json.dumps(report, indent=4))
        
        final_model.save_model("ml/event_intelligence/event_model.json")
        with open("ml/event_intelligence/training_report.json", "w") as f:
            json.dump(report, f, indent=4)
        print("\n[SUCCESS] K-Fold training complete.")

    except Exception as e:
        print("\n--- ERROR DURING TRAINING ---")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()
